In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s5e11_path = kagglehub.competition_download('playground-series-s5e11')

print('Data source import complete.')


In [ ]:
import numpy as np
import pandas as pd
import gc
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

# Install GPU-enabled libraries if missing
try:
    import xgboost
    import lightgbm
    import catboost
except ImportError:
    !pip install xgboost lightgbm catboost

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

SEED = 42

In [ ]:
def engineer_features(df):
    df = df.copy()

    # --- 1. Domain Knowledge Interactions ---
    # Standard Debt Load
    if 'annual_income' in df.columns and 'debt_to_income_ratio' in df.columns:
        df['total_debt'] = df['annual_income'] * df['debt_to_income_ratio']

    # --- 2. The "SHAP Diagnosis" (dricks77) ---
    # Flagging Low DTI but Risky Employment
    if 'debt_to_income_ratio' in df.columns and 'employment_status' in df.columns:
        is_risky_job = df['employment_status'].isin(['Unemployed', 'Student'])
        is_low_dti = df['debt_to_income_ratio'] < 0.15
        df['flag_low_dti_risky'] = (is_risky_job & is_low_dti).astype(int)

    # --- 3. Padé Approximants & Inverse Features (Laureano Arcanio / broccoli beef) ---
    # Models the "ceiling effect" where risk shoots up asymptotically
    risk_cols = ['credit_score', 'debt_to_income_ratio', 'loan_amount', 'interest_rate']

    # We need a scaler just for the Padé calculation (doesn't affect main data)
    mm = MinMaxScaler(feature_range=(-1, 1))

    for col in risk_cols:
        if col in df.columns:
            # Scale to -1 to 1 to prevent math explosions
            x = mm.fit_transform(df[[col]]).flatten()

            # Squared deviation from mean (Z-score-ish)
            mu = np.mean(x)
            z_sq = (x - mu)**2

            # Rational Function [2/2] Padé Approximant
            # Formula: (1 + 0.5z^2) / (1 + 0.3z^2)
            eps = 1e-6
            num = 1.0 + 0.5 * z_sq + 0.1 * (z_sq**2)
            den = 1.0 + 0.3 * z_sq + 0.05 * (z_sq**2) + eps

            df[f'{col}_pade_risk'] = num / den

            # Simple Reciprocal (broccoli beef suggestion)
            # Add small epsilon to avoid division by zero
            if df[col].min() == 0:
                df[f'{col}_inverse'] = 1 / (df[col] + 1.0)
            else:
                df[f'{col}_inverse'] = 1 / df[col]

    return df

In [ ]:

# ==========================================
# 2. DATA LOADING & PREP
# ==========================================
print("Loading Data...")
train = pd.read_csv("/kaggle/input/playground-series-s5e11/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s5e11/test.csv")

# Drop Junk
train = train.dropna(subset=['loan_paid_back'])
X = train.drop(columns=['id', 'loan_paid_back', 'gender', 'marital_status'])
y = train['loan_paid_back']
X_test_raw = test.drop(columns=['id', 'gender', 'marital_status'])

# Apply Engineering
print("Engineering Features...")
X = engineer_features(X)
X_test = engineer_features(X_test_raw)


Loading Data...
Engineering Features...


In [ ]:
# ==========================================
# 3. THE TRAINING ENGINE (GPU + Early Stopping)
# ==========================================
def train_model_oof(X, y, X_test, model_type='xgb', n_folds=5):
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    scores = []

    # --- FIX: Define Preprocessor INSIDE the function ---
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]),
         make_column_selector(dtype_include='number')),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                          ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
         make_column_selector(dtype_include=['object', 'category']))
    ])
    # -----------------------------------------------------

    print(f"\n🚀 Training {model_type.upper()} on GPU...")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr_raw, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_val_raw, y_val = X.iloc[val_idx], y.iloc[val_idx]

        # Transform Data
        preprocessor.fit(X_tr_raw)  # Now this line will work!
        X_tr = preprocessor.transform(X_tr_raw)
        X_val = preprocessor.transform(X_val_raw)
        X_test_trans = preprocessor.transform(X_test)

        # Define GPU Models
        if model_type == 'xgb':
            clf = XGBClassifier(
                n_estimators=10000, learning_rate=0.01, max_depth=6,
                subsample=0.8, colsample_bytree=0.8,
                early_stopping_rounds=100, eval_metric='auc',
                tree_method='hist', device='cuda',
                random_state=SEED
            )
            clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        elif model_type == 'lgbm':
            clf = LGBMClassifier(
                n_estimators=10000, learning_rate=0.01, num_leaves=31,
                subsample=0.8, colsample_bytree=0.8,
                early_stopping_rounds=100, metric='auc',
                device='gpu',
                random_state=SEED, verbose=-1
            )
            clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])

        elif model_type == 'cat':
            clf = CatBoostClassifier(
                iterations=10000, learning_rate=0.01, depth=6,
                eval_metric='AUC', early_stopping_rounds=100,
                task_type='GPU', devices='0:1', # Use both GPUs
                verbose=0, random_state=SEED
            )
            clf.fit(X_tr, y_tr, eval_set=(X_val, y_val))

        # Predict
        val_p = clf.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_p
        test_preds += clf.predict_proba(X_test_trans)[:, 1] / n_folds

        score = roc_auc_score(y_val, val_p)
        scores.append(score)
        print(f"Fold {fold+1}: {score:.5f}")

        del X_tr, X_val, clf
        gc.collect()

    print(f"✅ {model_type.upper()} Avg AUC: {np.mean(scores):.5f}")
    return oof_preds, test_preds

In [ ]:
# ==========================================
# 4. EXECUTION
# ==========================================

# A. Train the "Big 3"
oof_xgb, pred_xgb = train_model_oof(X, y, X_test, 'xgb')
oof_lgbm, pred_lgbm = train_model_oof(X, y, X_test, 'lgbm')
oof_cat, pred_cat = train_model_oof(X, y, X_test, 'cat')

# B. Save OOF Predictions (For Stacking analysis later if needed)
oof_df = pd.DataFrame({'xgb': oof_xgb, 'lgbm': oof_lgbm, 'cat': oof_cat, 'target': y})
oof_df.to_csv('oof_predictions.csv', index=False)

# C. Train Meta-Learner (Logistic Regression)
print("\n🥞 Training Meta-Learner (Stacking)...")
X_meta = oof_df[['xgb', 'lgbm', 'cat']]
X_meta_test = pd.DataFrame({'xgb': pred_xgb, 'lgbm': pred_lgbm, 'cat': pred_cat})

meta_model = LogisticRegression()
meta_model.fit(X_meta, y)

# D. Print Blending Weights
weights = meta_model.coef_[0]
print(f"Blending Weights -> XGB: {weights[0]:.2f}, LGBM: {weights[1]:.2f}, Cat: {weights[2]:.2f}")

# E. Final Submission
final_preds = meta_model.predict_proba(X_meta_test)[:, 1]

sub = pd.DataFrame({'id': test['id'], 'loan_paid_back': final_preds})
sub.to_csv('submission.csv', index=False)
print("✅ Submission Saved! Good luck!")


🚀 Training XGB on GPU...


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [19:11:59] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


Fold 1: 0.92422
Fold 2: 0.92412
Fold 3: 0.92236
Fold 4: 0.92354
Fold 5: 0.92272
✅ XGB Avg AUC: 0.92339

🚀 Training LGBM on GPU...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Fold 1: 0.92378
Fold 2: 0.92332
Fold 3: 0.92219
Fold 4: 0.92267
